# NVS Benchmark no Google Colab

[Abrir no Colab](https://colab.research.google.com/github/PedroHeinrichSP/TCC-Source-Code/blob/update/notebooks/nvs_benchmark_colab.ipynb) — clique para abrir e rodar o notebook no Google Colab.

Notebook focado em execucao no Colab: clone, setup, benchmark rapido, relatorio HTML, download e backup opcional no Drive.

## 📋 Como usar

1. **Execute as céulas na ordem** (Colab executa linearmente de cima para baixo)
2. **Célula 1 (esta)**: Documentação — sem ação necessária
3. **Célula 2**: Configure repositório e opções globais (REPO_URL, BRANCH, etc). Geralmente não muda.
4. **Célula 3**: Clone do repositório — não edite
5. **Célula 4**: Setup do ambiente — não edite
6. **Célula 5**: Verificação de GPU — não edite
7. **Célula 6**: Montagem opcional do Google Drive — edite `USE_GOOGLE_DRIVE` aqui se quiser backup
8. **⭐ Célula 7 (NOVA)**: Descoberta e seleção visual — **EDITE AQUI para escolher datasets, métodos, preset e modo de execução**. Caixas de seleção aparecerão se houver widgets disponíveis.
9. **Célula 8**: Download de datasets — executado automaticamente
10. **Célula 9**: Benchmark — usa as seleções da Célula 7. Não edite, apenas execute.
11. **Célula 10**: Relatório HTML — não edite
12. **Célula 11**: Exibição do relatório — não edite
13. **Célula 12**: Download de artefatos — não edite
14. **Célula 13**: Backup no Drive — não edite

## 🎯 O que editar e onde

| O que mudar | Onde | Como |
| --- | --- | --- |
| **URL do repositório ou branch** | Célula 2 | Mude `REPO_URL` ou `BRANCH` |
| **Método (nerf_static, gs_static, etc)** | Célula 7 | Selecione no widget ou edite `SELECTED_METHOD` |
| **Dataset (blender_synthetic, d_nerf, etc)** | Célula 7 | Selecione no widget ou edite `SELECTED_DATASET` |
| **Preset (smoke, quick, preview, standard, full)** | Célula 7 | Selecione no widget ou edite `SELECTED_PRESET` |
| **Modo de execução (quick_check ou full)** | Célula 7 | Selecione no widget ou edite `RUN_MODE` |
| **Modo estrito (resultados validados)** | Célula 7 | Marque/desmarque no widget ou edite `STRICT_RESULTS` |
| **Backup no Google Drive** | Célula 6 | Mude `USE_GOOGLE_DRIVE = True` ou `False` |
| **Gerar PDF do relatório** | Célula 7 | Marque/desmarque no widget ou edite `GENERATE_PDF` |

## ✅ Fluxo de execução

```
[Clone] → [Setup] → [GPU?] → [Drive?] → [Selecione] → [Download datasets] → [Benchmark] → [Relatório] → [Download]
```

**Tempo estimado**: 10–30 min (depende do preset e dataset escolhido).


In [ ]:
# Parametros globais de repositorio e caminhos (geralmente não edite)
REPO_URL = "https://github.com/PedroHeinrichSP/TCC-Source-Code.git"
REPO_DIR = "/content/TCC"
BRANCH = "update"
RUN_ID = "colab_quick"
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NVS_Benchmark"
DRIVE_DATA_DIR = f"{DRIVE_OUTPUT_DIR}/data"
DRIVE_ARTIFACTS_DIR = f"{DRIVE_OUTPUT_DIR}/artifacts"

# NOTA: Método, dataset, preset e modo são configurados na Célula 7 (Descoberta e Seleção Visual)
# Não edite aqui. Use a Célula 7 para mudar essas opções.

In [ ]:
# Clone do repositório
# Configurações estão na Célula 2: REPO_URL, BRANCH, REPO_DIR
print("=" * 70)
print(f"Clonando repositório: {REPO_URL} (branch: {BRANCH})")
print("=" * 70)

import os
import shutil
import subprocess
import sys

try:
    __import__("google.colab")
except Exception as exc:
    raise RuntimeError("Este notebook foi desenhado para Google Colab.") from exc

# Mude para um diretório seguro antes de remover REPO_DIR (evita erros no Colab)
os.chdir("/content") if os.path.exists("/content") else os.chdir(os.path.expanduser("~"))

if os.path.exists(REPO_DIR):
    print(f"Removendo pasta existente: {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

print("Clonando...")
clone_result = subprocess.run([
    "git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR
], text=True, capture_output=True)
if clone_result.stdout:
    print(clone_result.stdout)
if clone_result.stderr:
    print(clone_result.stderr)
if clone_result.returncode != 0:
    raise RuntimeError(f"Falha ao clonar o repositório (code={clone_result.returncode}).")

os.chdir(REPO_DIR)
print(f"\n✓ Projeto clonado em: {os.getcwd()}")

In [ ]:
# Setup do ambiente
print("=" * 70)
print("Instalando dependências...")
print("=" * 70)

pip_upgrade = subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], text=True, capture_output=True)
print(pip_upgrade.stdout)
if pip_upgrade.stderr:
    print(pip_upgrade.stderr)
if pip_upgrade.returncode != 0:
    raise RuntimeError(f"Falha ao atualizar pip (code={pip_upgrade.returncode}).")

pip_install = subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], text=True, capture_output=True)
print(pip_install.stdout)
if pip_install.stderr:
    print(pip_install.stderr)
if pip_install.returncode != 0:
    raise RuntimeError(f"Falha ao instalar o projeto em modo editável (code={pip_install.returncode}).")

print("\nVerificando instalação...")
status_result = subprocess.run([sys.executable, "-m", "nvs_benchmark.cli", "status"], text=True, capture_output=True)
print(status_result.stdout)
if status_result.stderr:
    print(status_result.stderr)
if status_result.returncode != 0:
    raise RuntimeError(f"Falha na verificação de status (code={status_result.returncode}).")

print("\n✓ Ambiente pronto.")

In [ ]:
# Montagem opcional do Google Drive
# Mude USE_GOOGLE_DRIVE para False se quiser rodar sem persistência no Drive
# (relatório, métricas e datasets podem ser copiados para /content/drive/MyDrive/NVS_Benchmark)
print("=" * 70)
print("Configuração: Google Drive")
print("=" * 70)
print(f"Backup no Drive: {'SIM' if USE_GOOGLE_DRIVE else 'NÃO'}")

if USE_GOOGLE_DRIVE:
    drive_mod = __import__("google.colab", fromlist=["drive"])
    drive = getattr(drive_mod, "drive")
    drive.mount("/content/drive")

    import os
    from pathlib import Path

    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_DATA_DIR).mkdir(parents=True, exist_ok=True)
    Path(DRIVE_ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

    print("✓ Drive montado com sucesso.")
    print(f"✓ Cache de dados: {DRIVE_DATA_DIR}")
    print(f"✓ Cache de artefatos: {DRIVE_ARTIFACTS_DIR}")
else:
    print("  (Para ativar, mude USE_GOOGLE_DRIVE = True na Célula 2)")
print()

In [ ]:
# Verificação de GPU no Colab
print("=" * 70)
print("Verificação de Hardware")
print("=" * 70)

import torch
cuda_available = torch.cuda.is_available()
print(f"CUDA disponível: {'SIM ✓' if cuda_available else 'NÃO ✗'}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ GPU não detectada. O benchmark rodará em CPU (mais lento).")
print()

In [ ]:
# 🎯 Descoberta, Seleção Visual de Datasets e Funções de Suporte
# Edite aqui para escolher qual método, dataset, preset e modo executar

from pathlib import Path
import shutil
import subprocess


def _project_root() -> Path:
    """Resolve a raiz do projeto a partir do cwd atual do notebook."""
    cwd = Path.cwd()
    for candidate in (cwd, cwd.parent):
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    return cwd


def discover_available_datasets() -> dict[str, str]:
    """Descobre datasets disponíveis em data/ com seus caminhos raiz."""
    available: dict[str, str] = {}
    data_dir = _project_root() / "data"

    markers = {
        "blender_synthetic": ["transforms_train.json"],
        "d_nerf": ["transforms_train.json"],
        "mipnerf360": ["poses_bounds.npy"],
        "tanks_and_temples": ["transforms_train.json", "poses_bounds.npy"],
        "custom": ["transforms_train.json", "poses_bounds.npy"],
    }

    if not data_dir.exists():
        print("[info] data/ não existe ainda. Datasets serão instalados na próxima célula.")
        return available

    for dataset_name, marker_list in markers.items():
        for marker in marker_list:
            for found_path in data_dir.rglob(marker):
                parent = found_path.parent
                if dataset_name == "mipnerf360":
                    path_str = str(parent).lower()
                    if "mipnerf360" not in path_str and "360_v2" not in path_str:
                        continue
                available[dataset_name] = str(parent)
                break
            if dataset_name in available:
                break

    return available


def discover_available_methods() -> list[str]:
    """Descobre métodos disponíveis no registry."""
    try:
        from nvs_benchmark.methods import build_registry_with_all_methods

        registry = build_registry_with_all_methods()
        if hasattr(registry, "list_ids"):
            return registry.list_ids()
        if hasattr(registry, "keys"):
            return list(registry.keys())
    except Exception as exc:
        print(f"[warn] Falha ao descobrir métodos: {exc}")

    return ["nerf_static", "nerf_dynamic", "gs_static", "gs_dynamic"]


def resolve_dataset_root(dataset_name: str, discovered: dict[str, str]) -> str:
    """Resolve a raiz do dataset com fallback explícito ancorado no projeto."""
    project_root = _project_root()
    fallback_paths = {
        "blender_synthetic": project_root / "data" / "blender_synthetic" / "nerf_synthetic" / "lego",
        "d_nerf": project_root / "data" / "d_nerf" / "bonsai",
        "mipnerf360": project_root / "data" / "mipnerf360" / "bicycle",
        "tanks_and_temples": project_root / "data" / "tanks_and_temples" / "Barn",
    }

    root = Path(discovered.get(dataset_name, fallback_paths.get(dataset_name, project_root / "data" / dataset_name)))
    if not root.is_absolute():
        root = project_root / root
    if not root.exists():
        print(f"[warn] Dataset '{dataset_name}' não encontrado em: {root}")
    return str(root)


def sync_dir_if_exists(source: Path, target: Path) -> bool:
    """Copia um diretório quando existe e o destino ainda não está preenchido."""
    if not source.exists():
        return False
    if target.exists() and any(target.iterdir()):
        return True
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        shutil.rmtree(target, ignore_errors=True)
    shutil.copytree(source, target)
    return True


def run_logged(cmd: list[str], *, check: bool = True, label: str | None = None) -> subprocess.CompletedProcess:
    """Executa subprocesso e imprime stdout/stderr no notebook."""
    if label:
        print(f"\n[{label}]")
    print(f"$ {' '.join(cmd)}")
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Comando falhou com code={completed.returncode}: {' '.join(cmd)}")
    return completed


# ============================================================================
# DEFAULTS (edite estes se os widgets não funcionarem ou para mudar valores)
# ============================================================================
SELECTED_METHOD = "nerf_static"      # Ex: "nerf_static", "gs_static", "gs_dynamic", "nerf_dynamic"
SELECTED_DATASET = "blender_synthetic"  # Ex: "blender_synthetic", "d_nerf", "mipnerf360", "tanks_and_temples"
SELECTED_PRESET = "quick"            # Ex: "smoke", "quick", "preview", "standard", "full"
RUN_MODE = "full"                    # "full" ou "quick_check" para teste rápido
APPLY_COMPATIBILITY_FILTER = True     # Filtrar combos incompatíveis
STRICT_RESULTS = True                 # Validar resultados antes de incluir no relatório
GENERATE_PDF = False                  # Gerar PDF além de HTML
MIN_REQUIRED_PAIRS = 1                # Número mínimo de pares de camera-imagem requerido


def _print_selection() -> None:
    print("=" * 70)
    print("SELEÇÃO ATUALIZADA")
    print("=" * 70)
    print(f"Método:           {SELECTED_METHOD}")
    print(f"Dataset:          {SELECTED_DATASET}")
    print(f"Caminho:          {SELECTED_DATASET_ROOT}")
    print(f"Preset:           {SELECTED_PRESET}")
    print(f"Modo:             {RUN_MODE}")
    print(f"Modo estrito:     {STRICT_RESULTS}")
    print(f"Gerar PDF:        {GENERATE_PDF}")
    print(f"Filtro compat:    {APPLY_COMPATIBILITY_FILTER}")
    print("=" * 70)


def update_selection(*_args) -> None:
    """Sincroniza widgets com variáveis globais usadas nas próximas células."""
    global SELECTED_METHOD, SELECTED_DATASET, SELECTED_PRESET
    global RUN_MODE, APPLY_COMPATIBILITY_FILTER, STRICT_RESULTS, GENERATE_PDF
    global SELECTED_DATASET_ROOT

    SELECTED_METHOD = method_dropdown.value
    SELECTED_DATASET = dataset_dropdown.value
    SELECTED_PRESET = preset_dropdown.value
    RUN_MODE = "quick_check" if quick_check_checkbox.value else "full"
    APPLY_COMPATIBILITY_FILTER = compat_checkbox.value
    STRICT_RESULTS = strict_checkbox.value
    GENERATE_PDF = pdf_checkbox.value
    SELECTED_DATASET_ROOT = resolve_dataset_root(SELECTED_DATASET, available_datasets)

    status_output.clear_output()
    with status_output:
        _print_selection()


# Descobre datasets e métodos
available_datasets = discover_available_datasets()
available_methods = discover_available_methods()

print("=" * 70)
print("DESCOBERTA: Datasets e Métodos Disponíveis")
print("=" * 70)
print(f"\nDatasets disponíveis em data/: {list(available_datasets.keys()) if available_datasets else '[nenhum]'}")
print(f"Métodos disponíveis: {available_methods}")
print()

# Seleções padrão
SELECTED_DATASET_ROOT = resolve_dataset_root(SELECTED_DATASET, available_datasets)

# ============================================================================
# INTERFACE COM WIDGETS (Colab / IPython)
# ============================================================================
USE_WIDGETS = False
try:
    from ipywidgets import Checkbox, Dropdown, HBox, VBox, Label, Output
    from IPython.display import display
    USE_WIDGETS = True
except ImportError:
    print("[info] ipywidgets não disponível. Use os valores defaults abaixo ou edite as variáveis acima.")

method_dropdown = None
dataset_dropdown = None
preset_dropdown = None
quick_check_checkbox = None
strict_checkbox = None
pdf_checkbox = None
compat_checkbox = None
status_output = None

if USE_WIDGETS and (available_datasets or available_methods):
    print("\n" + "=" * 70)
    print("SELEÇÃO VISUAL (clique nas opções abaixo)")
    print("=" * 70 + "\n")

    method_dropdown = Dropdown(
        options=available_methods,
        value=SELECTED_METHOD if SELECTED_METHOD in available_methods else available_methods[0],
        description="Método:",
        disabled=False,
    )

    dataset_dropdown = Dropdown(
        options=list(available_datasets.keys()) if available_datasets else ["blender_synthetic"],
        value=SELECTED_DATASET if SELECTED_DATASET in available_datasets else (list(available_datasets.keys())[0] if available_datasets else "blender_synthetic"),
        description="Dataset:",
        disabled=not bool(available_datasets),
    )

    preset_dropdown = Dropdown(
        options=["smoke", "quick", "preview", "standard", "full"],
        value=SELECTED_PRESET,
        description="Preset:",
        disabled=False,
    )

    quick_check_checkbox = Checkbox(
        value=(RUN_MODE == "quick_check"),
        description="Teste rápido (quick_check)?",
        indent=False,
    )

    strict_checkbox = Checkbox(
        value=STRICT_RESULTS,
        description="Modo estrito (validar resultados)?",
        indent=False,
    )

    pdf_checkbox = Checkbox(
        value=GENERATE_PDF,
        description="Gerar PDF do relatório?",
        indent=False,
    )

    compat_checkbox = Checkbox(
        value=APPLY_COMPATIBILITY_FILTER,
        description="Filtrar combos incompatíveis?",
        indent=False,
    )

    status_output = Output()

    for widget in [method_dropdown, dataset_dropdown, preset_dropdown, quick_check_checkbox, strict_checkbox, pdf_checkbox, compat_checkbox]:
        widget.observe(update_selection, names="value")

    display(VBox([
        Label("📌 Escolha as opções abaixo e execute a célula:"),
        HBox([method_dropdown, dataset_dropdown]),
        HBox([preset_dropdown]),
        HBox([quick_check_checkbox]),
        strict_checkbox,
        pdf_checkbox,
        compat_checkbox,
        status_output,
    ]))

    update_selection()
else:
    _print_selection()


DESCOBERTA: Datasets e Métodos Disponíveis

Datasets disponíveis em data/: ['blender_synthetic', 'd_nerf', 'tanks_and_temples', 'custom']
Métodos disponíveis: ['gs_dynamic', 'gs_static', 'nerf_dynamic', 'nerf_static']

[info] ipywidgets não disponível. Use os valores defaults abaixo ou edite as variáveis acima.
SELEÇÃO ATUALIZADA
Método:           nerf_static
Dataset:          blender_synthetic
Caminho:          c:\Users\Admin\Projetos\TCC\data\blender_synthetic\nerf_synthetic\lego
Preset:           quick
Modo:             full
Modo estrito:     True
Gerar PDF:        False
Filtro compat:    True


In [ ]:
# Download de datasets via catálogo com cache no Drive
# Se USE_GOOGLE_DRIVE=True, tenta restaurar datasets já baixados antes de instalar de novo.
print("=" * 70)
print("Preparando datasets...")
print("=" * 70)
print(f"Dataset selecionado para execução: {SELECTED_DATASET}")
print()

from pathlib import Path

local_data_dir = Path("./data")
drive_data_dir = Path(DRIVE_DATA_DIR)

cache_restored = False
if USE_GOOGLE_DRIVE:
    if sync_dir_if_exists(drive_data_dir, local_data_dir):
        print(f"✓ Cache de datasets restaurado de: {drive_data_dir}")
        cache_restored = True
    else:
        print("[info] Nenhum cache de datasets encontrado no Drive. Será feita instalação local.")

# Verificar quais datasets estão disponíveis após cache restore
available_for_run = discover_available_datasets()
if SELECTED_DATASET in available_for_run:
    print(f"✓ Dataset '{SELECTED_DATASET}' já disponível em: {available_for_run[SELECTED_DATASET]}")
    print("  (Pulando download automático)")
else:
    print(f"⚠ Dataset '{SELECTED_DATASET}' não encontrado. Tentando instalar...")
    
    install_cmd = [
        sys.executable, "-m", "nvs_benchmark.cli", "install",
        "--catalog-file", "./configs/install_catalog.json",
        "--only", "datasets",
        "--execute"
    ]
    
    install_result = run_logged(install_cmd, label="datasets-install", check=False)
    
    # Validar novamente após install
    available_for_run = discover_available_datasets()
    if SELECTED_DATASET not in available_for_run:
        print(f"\n⚠ AVISO: Dataset '{SELECTED_DATASET}' ainda não disponível após install.")
        print("  Você pode:")
        print(f"  1. Tentar novamente (alguns downloads são intermitentes)")
        print(f"  2. Baixar manualmente (ver URLs em ./configs/install_catalog.json)")
        print(f"  3. Selecionar um dataset diferente (veja lista acima)")

if USE_GOOGLE_DRIVE and local_data_dir.exists():
    sync_dir_if_exists(local_data_dir, drive_data_dir)
    print(f"✓ Cache de datasets sincronizado para: {drive_data_dir}")

print("\n✓ Datasets preparados.")

In [ ]:
# Benchmark (usa seleções da Célula 7)
print("=" * 70)
print(f"Executando benchmark: {SELECTED_METHOD} x {SELECTED_DATASET}")
print("=" * 70)

from pathlib import Path

snapshot_file = f"./artifacts/metrics/{RUN_ID}.json"

# Validação rápida do dataset antes do benchmark pesado
preflight_cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "dataset-check",
    "--dataset", SELECTED_DATASET,
    "--root", SELECTED_DATASET_ROOT,
    "--split", "train",
]
preflight = run_logged(preflight_cmd, label="dataset-preflight", check=False)
if preflight.returncode != 0:
    raise RuntimeError(
        f"dataset-check falhou para {SELECTED_DATASET} em {SELECTED_DATASET_ROOT}. "
        "Corrija a raiz do dataset na Célula 7 antes de continuar."
    )

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "method-run",
    "--method", SELECTED_METHOD,
    "--dataset", SELECTED_DATASET,
    "--root", SELECTED_DATASET_ROOT,
    "--split", "train",
    "--preset", SELECTED_PRESET,
    "--output-dir", "./artifacts",
    "--log-dir", "./logs",
    "--compute-metrics",
    "--snapshot-file", snapshot_file,
    "--append-snapshot",
]

if STRICT_RESULTS:
    cmd.extend(["--strict-results", "--min-required-pairs", str(MIN_REQUIRED_PAIRS)])

result = run_logged(cmd, label="method-run", check=False)
if result.returncode != 0:
    raise RuntimeError(
        f"method-run falhou para {SELECTED_METHOD} x {SELECTED_DATASET} (code={result.returncode}). "
        "Veja os logs acima para a causa exata."
    )

print(f"\n✓ Snapshot gerado em: {snapshot_file}")

In [ ]:
# Gerar relatorio HTML (usa seleções da Célula 7)
print("=" * 70)
print("Gerando relatório HTML...")
print("=" * 70)

report_name = f"{RUN_ID}_{SELECTED_METHOD}_{SELECTED_DATASET}_report"

cmd = [
    sys.executable, "-m", "nvs_benchmark.cli", "report-generate",
    "--snapshot-file", snapshot_file,
    "--output-dir", "./artifacts/reports",
    "--report-name", report_name,
    "--log-dir", "./logs",
]

if not GENERATE_PDF:
    cmd.append("--no-pdf")

if STRICT_RESULTS:
    cmd.extend(["--strict-snapshot", "--min-methods", "1", "--require-finite-metrics"])

report_result = run_logged(cmd, label="report-generate", check=False)
if report_result.returncode != 0:
    raise RuntimeError("Falha ao gerar o relatório consolidado.")

report_html = f"./artifacts/reports/{report_name}.html"

if USE_GOOGLE_DRIVE:
    sync_dir_if_exists(Path("./artifacts"), Path(DRIVE_ARTIFACTS_DIR))
    print(f"✓ Artefatos sincronizados para: {DRIVE_ARTIFACTS_DIR}")

print(f"\n✓ Relatório HTML gerado: {report_html}")

In [ ]:
# Exibir o relatório no notebook
print("=" * 70)
print("Carregando relatório...")
print("=" * 70)

from IPython.display import IFrame, display
import os

if not os.path.exists(report_html):
    print(f"✗ Arquivo não encontrado: {report_html}")
    print("  Verifique se a célula anterior executou sem erros.")
else:
    print(f"✓ Abrindo: {report_html}\n")
    display(IFrame(src=report_html, width=1200, height=700))

In [ ]:
# Compactar e baixar artefatos
print("=" * 70)
print("Preparando download de artefatos...")
print("=" * 70)

import pathlib
colab_files_mod = __import__("google.colab", fromlist=["files"])
files = getattr(colab_files_mod, "files")

zip_path = "/content/nvs_benchmark_artifacts"
print(f"Compactando {REPO_DIR}/artifacts...")
archive_file = shutil.make_archive(zip_path, "zip", REPO_DIR, "artifacts")
print(f"Arquivo gerado: {archive_file}")

if USE_GOOGLE_DRIVE:
    drive_archive_dir = Path(DRIVE_ARTIFACTS_DIR) / "archives"
    drive_archive_dir.mkdir(parents=True, exist_ok=True)
    drive_archive_file = drive_archive_dir / Path(archive_file).name
    shutil.copy2(archive_file, drive_archive_file)
    print(f"✓ ZIP salvo no Drive em: {drive_archive_file}")

if pathlib.Path(archive_file).exists():
    print(f"Tamanho: {pathlib.Path(archive_file).stat().st_size / (1024*1024):.1f} MB")
    print("\n↓ Iniciando download...")
    files.download(archive_file)
else:
    print("✗ Arquivo não encontrado!")

In [ ]:
# Backup opcional dos artefatos no Google Drive
print("=" * 70)
print("Backup de resultados")
print("=" * 70)

if USE_GOOGLE_DRIVE:
    print(f"Backup: SIM → {DRIVE_OUTPUT_DIR}")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    metrics_dst = os.path.join(DRIVE_OUTPUT_DIR, "metrics")
    reports_dst = os.path.join(DRIVE_OUTPUT_DIR, "reports")

    if os.path.exists(metrics_dst):
        shutil.rmtree(metrics_dst)
    if os.path.exists(reports_dst):
        shutil.rmtree(reports_dst)

    print(f"Copiando métricas...")
    shutil.copytree("./artifacts/metrics", metrics_dst)
    print(f"Copiando relatórios...")
    shutil.copytree("./artifacts/reports", reports_dst)

    drive_data_dst = Path(DRIVE_DATA_DIR)
    if Path("./data").exists():
        if drive_data_dst.exists():
            shutil.rmtree(drive_data_dst)
        print("Sincronizando datasets para o Drive...")
        shutil.copytree("./data", drive_data_dst)

    print(f"✓ Backup concluido em: {DRIVE_OUTPUT_DIR}")
else:
    print("Backup: NÃO")
    print("  Para fazer backup no Drive, mude USE_GOOGLE_DRIVE = True na Célula 2")